# Full-set re-evaluation (matches docx methodology)

The reference docx `yolo_research_final_v1.docx` reports v11 metrics that were
computed by running `YOLO.val()` on the **full 1,026-image dataset** — not a
held-out split. To add v12 rows to that docx apples-to-apples, this notebook
re-evaluates each v12 `best.pt` the same way.

**What you need on Drive first:**
1. `MyDrive/yolo-pipeline/runs/yolo12{n,s,m,l}_<timestamp>/` — the trained runs
2. `MyDrive/yolo-pipeline/full_dataset/` — a folder with `images/` + `labels/` +
   `classes.txt` (copy of `report_generation/data/` from local). Upload this once.

**Output:** each run folder gets a `full_eval/per_class.json` with per-class
Precision/Recall/mAP@0.5/mAP@0.5:0.95 on the full set — the same numbers
reported in the docx for v11.

In [ ]:
REPO_URL    = 'https://github.com/tahmid013/yolo.git'
REPO_BRANCH = 'main'

# Which run folders on Drive to re-evaluate. Edit as needed.
# Full names as they appear under MyDrive/yolo-pipeline/runs/
RUN_NAMES = [
    'yolo12n_20260725-111453',
    # 'yolo12s_<timestamp>',   # add after training
    # 'yolo12m_<timestamp>',
    # 'yolo12l_<timestamp>',
]

# Drive path to the full labelled dataset (contains images/, labels/, classes.txt)
FULL_DATASET_DRIVE = '/content/drive/MyDrive/yolo-pipeline/full_dataset'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
import shutil, os, sys, subprocess
os.chdir('/content')
if os.path.isdir('/content/code'):
    shutil.rmtree('/content/code')
res = subprocess.run(
    ['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, '/content/code'],
    capture_output=True, text=True,
)
if res.returncode != 0:
    print('git stderr:', res.stderr); raise RuntimeError('git clone failed')
os.chdir('/content/code')
if '/content/code' not in sys.path:
    sys.path.insert(0, '/content/code')
for mod in [m for m in list(sys.modules) if m == 'pipeline' or m.startswith('pipeline.')]:
    del sys.modules[mod]
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
full_ds = Path(FULL_DATASET_DRIVE)
if not full_ds.is_dir():
    raise FileNotFoundError(
        f'Full dataset not found at {full_ds}. Upload report_generation/data/ '
        f'contents (images/ + labels/ + classes.txt) to Drive at that path.'
    )
assert (full_ds / 'images').is_dir(), f'images/ subfolder missing in {full_ds}'
assert (full_ds / 'labels').is_dir(), f'labels/ subfolder missing in {full_ds}'
assert (full_ds / 'classes.txt').is_file(), f'classes.txt missing in {full_ds}'
print(f'full dataset OK: {len(list((full_ds / "images").glob("*")))} images')

In [ ]:
from pipeline import full_eval
from pathlib import Path

DRIVE_RUNS = Path('/content/drive/MyDrive/yolo-pipeline/runs')
results = {}

for name in RUN_NAMES:
    run_dir = DRIVE_RUNS / name
    if not run_dir.is_dir():
        print(f'SKIP {name}: not found at {run_dir}')
        continue
    payload = full_eval.run(
        run_dir=run_dir,
        source_dataset_dir=full_ds,
        drive_runs_dir=DRIVE_RUNS,
        push_to_drive=False,   # run_dir already IS on Drive; no separate copy needed
    )
    results[name] = payload['overall']

print('\n=== Summary (mAP@0.5 on full 1026-image set) ===')
for name, o in results.items():
    print(f"  {name}: mAP50={o['mAP50']:.4f}  mAP50-95={o['mAP50_95']:.4f}  P={o['precision']:.3f}  R={o['recall']:.3f}")